© 2026 Mechanome Bio. Proprietary and confidential.

# Nuclear shape gSEM — step 1 of 3: measure nuclear shape and export for R

**Kernel: Python** (the same environment you use for the segmentation-Pipeline-Benchmarking notebooks — it needs `spatialdata`, `anndata`, `shapely`, `scipy`, `pandas`). Run on Windows as normal.

This notebook prepares everything the R gSEM notebook needs, for each tissue:

1. Reads the SpatialData zarr store **in place** from `../segmentation-Pipeline-Benchmarking/data` (nothing there is copied, moved or changed).
2. Measures six nuclear shape metrics on every Xenium nucleus outline, using exact polygon geometry.
3. Applies the nucleus and cell quality filters listed below.
4. Writes a per-cell table and a cells × genes count matrix to `outputs/<tissue>/prep/`.

| Notebook | Kernel | Where to run | What it does |
|---|---|---|---|
| `nuclear_shape_prep.ipynb` (this one) | Python | Windows | measure shapes, filter, export |
| `nuclear_shape_gsem.ipynb` | R (`mechmap-stats`) | **WSL** | gSEM per metric, saves every table needed for plots |
| `nuclear_shape_plots.ipynb` | Python | Windows | house-style figures from the saved tables |

## Assumptions and choices built into this notebook

| Topic | Choice | Why / note |
|---|---|---|
| Tissues | pancreas and lung, run separately | `DATASETS_TO_RUN` in the settings cell |
| Nucleus outlines | Xenium `nucleus_boundaries` from the zarr | keyed by Xenium `cell_id`, so they join directly to the count table |
| Expression | the zarr count table (`tables/table`, from `cell_feature_matrix.h5`) | 10x has already applied its transcript quality filter (Q ≥ 20) and assigned transcripts to cells (whole cell, not nucleus only). Only "Gene Expression" features are kept; control probes / codewords are removed |
| Coordinates | microns | the zarr stores shapes and centroids in µm, so `perimeter` is in µm |
| Measurement | exact polygon geometry (shapely), not pixels | see `src/nuclear_shape_metrics.py`; checked by `tests/test_nuclear_shape_metrics.py` |
| `solidity` | area / convex hull area | |
| `extent` | area / **axis-aligned** bounding box area | matches skimage; depends on the nucleus' rotation on the slide |
| `shape_index` | perimeter / √area | |
| `circularity` | 4π·area / perimeter² | |
| `perimeter` | outline length (µm) | strongly tied to size; the gSEM notebook smooths out log **cell** area only (not nuclear area), as chosen |
| `axis_ratio` | minor / major axis of the equivalent ellipse (second moments of area) | in (0, 1]; 1 = round |
| Several nucleus outlines for one cell | keep the largest | `MULTIPLE_NUCLEI_RULE` |
| Outline made of several pieces | keep the largest piece; holes are filled | |
| Invalid outlines | dropped (shapely `is_valid` false, or zero area) | |
| Small nuclei | dropped below the 1st percentile of nuclear area (auto floor, per tissue) | `NUCLEAR_AREA_FLOOR_PERCENTILE` |
| Low-count cells | dropped below 10 gene counts | `MINIMUM_COUNTS_PER_CELL` |
| Cells with no nucleus outline | dropped | |
| Spatial position used by gSEM | the Xenium **cell** centroid (`x_centroid`, `y_centroid`) | as MB_05.2 uses cell centroids |
| Cell area (for normalisation and the area covariate) | Xenium `cell_area` from the zarr table | as MB_05.2 |
| Cell types (pancreas only) | `cell_type` column of `pancreas_s0005_adata_annotated.h5ad` | only cells that passed MechanoMap are annotated; others are left blank and are simply not used in the cell-type step |


In [1]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 1: imports and folder locations.
import sys
import json
import datetime
import platform
from pathlib import Path

import numpy
import pandas
import scipy.sparse
import scipy.io
import anndata
import spatialdata

# This notebook lives in nuclear_shape/notebooks, so the repository folder is one level up.
REPOSITORY_FOLDER = Path.cwd().parent
sys.path.insert(0, str(REPOSITORY_FOLDER / "src"))
import nuclear_shape_metrics as metrics  # the exact polygon measurements (src/nuclear_shape_metrics.py)

# The input data is READ IN PLACE from the segmentation benchmarking repository, which sits
# next to this repository in C:\MechanomeBioRepos. Nothing in that folder is written to.
INPUT_DATA_FOLDER = REPOSITORY_FOLDER.parent / "segmentation-Pipeline-Benchmarking" / "data"
OUTPUT_FOLDER = REPOSITORY_FOLDER / "outputs"

PROPRIETARY_NOTICE = "© 2026 Mechanome Bio. Proprietary and confidential."

print("Repository folder :", REPOSITORY_FOLDER)
print("Input data folder :", INPUT_DATA_FOLDER, "(exists)" if INPUT_DATA_FOLDER.exists() else "(NOT FOUND - check the path)")
print("Output folder     :", OUTPUT_FOLDER)

Repository folder : c:\MechanomeBioRepos\nuclear_shape
Input data folder : c:\MechanomeBioRepos\segmentation-Pipeline-Benchmarking\data (exists)
Output folder     : c:\MechanomeBioRepos\nuclear_shape\outputs


In [2]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 2: settings. Everything you might want to change is in this cell.

# Which tissues to process. Each one gets its own folder: outputs/<tissue name>/prep/
DATASETS_TO_RUN = ["pancreas", "lung"]

# Where each tissue's data lives inside INPUT_DATA_FOLDER.
# cell_type_file / cell_type_column: optional per-cell cell-type labels (None = no labels).
DATASETS = {
    "pancreas": {
        "zarr_store": "Xenium_V1_human_Pancreas_FFPE_outs.zarr",
        "cell_type_file": "pancreas_s0005_adata_annotated.h5ad",
        "cell_type_column": "cell_type",
    },
    "lung": {
        "zarr_store": "Xenium_V1_humanLung_Cancer_FFPE_outs.zarr",
        "cell_type_file": None,
        "cell_type_column": None,
    },
}

# Some cells have more than one nucleus outline. "largest" keeps the biggest one;
# "drop" removes any cell with more than one nucleus outline.
MULTIPLE_NUCLEI_RULE = "largest"

# Nuclei smaller than this percentile of nuclear area (worked out separately for each tissue)
# are dropped, to remove fragments and segmentation debris. Set to None to keep all sizes.
NUCLEAR_AREA_FLOOR_PERCENTILE = 1.0

# Cells with fewer than this many gene counts (summed over all genes) are dropped.
MINIMUM_COUNTS_PER_CELL = 10

# Feature names that are NOT real genes (negative controls, blank / unassigned codewords).
# Anything whose name starts with one of these is removed from the count matrix.
NON_GENE_PREFIXES = ("NegControl", "UnassignedCodeword", "BLANK", "DeprecatedCodeword", "Intergenic")

# Prefix that MechanoMap adds to cell names in the annotated AnnData file, e.g. "R0C0_Y0_X0_".
TILE_PREFIX_PATTERN = r"^R\d+C\d+_Y\d+_X\d+_"

print("Tissues to process:", DATASETS_TO_RUN)

Tissues to process: ['pancreas', 'lung']


In [3]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 3: helper functions used by the next cell.


def read_store(zarr_path):
    '''Read only the shapes and table from a SpatialData zarr store (skips the large image).'''
    try:
        return spatialdata.read_zarr(zarr_path, selection=("shapes", "tables"))
    except TypeError:
        # Older spatialdata versions have no "selection" option; the image is read lazily anyway.
        return spatialdata.read_zarr(zarr_path)


def gene_count_table(table):
    '''
    Return (counts, cell_ids, gene_names) from the zarr's AnnData count table.

    counts is a sparse cells x genes matrix (most entries are zero, so only the non-zero
    entries are stored). Control probes and codewords are removed so that only real genes remain.
    '''
    keep_feature = numpy.ones(table.n_vars, dtype=bool)
    if "feature_types" in table.var.columns:
        keep_feature &= (table.var["feature_types"].astype(str) == "Gene Expression").to_numpy()
    feature_names = table.var_names.astype(str)
    keep_feature &= ~feature_names.str.startswith(NON_GENE_PREFIXES)

    counts = scipy.sparse.csr_matrix(table.X)[:, keep_feature]

    # The counts should be whole numbers (they are stored as float32). If they are not, this
    # table has already been normalised and must not be used as raw counts.
    non_zero_values = counts.data
    if non_zero_values.size and not numpy.allclose(non_zero_values, numpy.round(non_zero_values)):
        raise ValueError("The zarr count table does not contain whole-number counts - it looks normalised.")
    counts = counts.astype(numpy.int32)

    cell_ids = table.obs["cell_id"].astype(str).to_numpy() if "cell_id" in table.obs.columns \
        else table.obs_names.astype(str).to_numpy()
    return counts, cell_ids, feature_names[keep_feature].to_numpy()


def nucleus_outlines_by_cell(nucleus_shapes):
    '''One row per nucleus outline with its Xenium cell_id (stored as a column or as the index).'''
    if "cell_id" in nucleus_shapes.columns:
        nucleus_cell_ids = nucleus_shapes["cell_id"].astype(str).to_numpy()
    else:
        nucleus_cell_ids = nucleus_shapes.index.astype(str).to_numpy()
    return pandas.DataFrame({"cell_id": nucleus_cell_ids, "geometry": nucleus_shapes.geometry.to_numpy()})


def write_csv_with_notice(data_frame, path, **to_csv_options):
    '''Write a CSV whose first line is the proprietary notice (read back with comment="#").'''
    with open(path, "w", encoding="utf-8", newline="") as output_file:
        output_file.write(f"# {PROPRIETARY_NOTICE}\n")
        data_frame.to_csv(output_file, **to_csv_options)


def write_list_with_notice(values, path):
    '''Write one value per line, after a proprietary-notice line that starts with "#".'''
    with open(path, "w", encoding="utf-8", newline="\n") as output_file:
        output_file.write(f"# {PROPRIETARY_NOTICE}\n")
        output_file.write("\n".join(map(str, values)) + "\n")


print("helper functions defined")

helper functions defined


In [4]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# CELL 4: for each tissue - measure nuclei, filter, and export for the R gSEM notebook.

for dataset_name in DATASETS_TO_RUN:
    settings = DATASETS[dataset_name]
    zarr_path = INPUT_DATA_FOLDER / settings["zarr_store"]
    prep_folder = OUTPUT_FOLDER / dataset_name / "prep"
    prep_folder.mkdir(parents=True, exist_ok=True)
    print(f"\n========== {dataset_name}: {zarr_path.name} ==========")

    # Keep a running count of cells at each filtering step, saved at the end as a QC table.
    quality_control_steps = []

    def record_step(step_description, number_of_cells):
        quality_control_steps.append({"step": step_description, "cells_remaining": int(number_of_cells)})
        print(f"  {step_description:<55s} {number_of_cells:>9,d} cells")

    # ---- 1. Read the count table and the nucleus outlines ------------------------------------
    spatial_data = read_store(zarr_path)
    table = spatial_data.tables["table"]
    counts, table_cell_ids, gene_names = gene_count_table(table)
    cell_information = table.obs.copy()
    cell_information.index = table_cell_ids
    for required_column in ["x_centroid", "y_centroid", "cell_area"]:
        if required_column not in cell_information.columns:
            raise KeyError(f"The zarr table has no '{required_column}' column; found {list(cell_information.columns)}")
    print(f"  count table: {counts.shape[0]:,} cells x {counts.shape[1]} genes (control features removed)")
    record_step("cells in the zarr count table", counts.shape[0])

    nuclei = nucleus_outlines_by_cell(spatial_data.shapes["nucleus_boundaries"])
    nuclei = nuclei[nuclei["cell_id"].isin(set(table_cell_ids))]
    record_step("cells with at least one nucleus outline", nuclei["cell_id"].nunique())

    # ---- 2. Clean the outlines and drop invalid ones -----------------------------------------
    nuclei["geometry"] = [metrics.keep_largest_piece(shape) for shape in nuclei["geometry"]]
    usable = numpy.array([metrics.is_usable_polygon(shape) for shape in nuclei["geometry"]])
    print(f"  invalid / zero-area nucleus outlines dropped: {(~usable).sum():,}")
    nuclei = nuclei[usable].copy()
    record_step("cells with at least one valid nucleus outline", nuclei["cell_id"].nunique())

    # ---- 3. One nucleus per cell -------------------------------------------------------------
    nuclei["outline_area"] = [shape.area for shape in nuclei["geometry"]]
    nuclei_per_cell = nuclei.groupby("cell_id")["outline_area"].transform("size")
    print(f"  cells with more than one valid nucleus outline: {nuclei.loc[nuclei_per_cell > 1, 'cell_id'].nunique():,}")
    if MULTIPLE_NUCLEI_RULE == "largest":
        nuclei = nuclei.sort_values("outline_area", ascending=False).drop_duplicates("cell_id")
    elif MULTIPLE_NUCLEI_RULE == "drop":
        nuclei = nuclei[nuclei_per_cell == 1]
    else:
        raise ValueError("MULTIPLE_NUCLEI_RULE must be 'largest' or 'drop'")
    record_step(f"cells after the '{MULTIPLE_NUCLEI_RULE}' multiple-nuclei rule", len(nuclei))

    # ---- 4. Measure the six shape metrics ----------------------------------------------------
    measurements = pandas.DataFrame(
        [metrics.measure_nucleus(shape) for shape in nuclei["geometry"]],
        index=nuclei["cell_id"].to_numpy())
    measurements["number_of_nucleus_outlines"] = nuclei_per_cell.loc[nuclei.index].to_numpy()
    measurements = measurements.dropna(subset=metrics.SHAPE_METRIC_NAMES)
    record_step("cells with all six metrics measurable", len(measurements))

    # ---- 5. Nuclear area floor ---------------------------------------------------------------
    nuclear_area_floor = None
    if NUCLEAR_AREA_FLOOR_PERCENTILE is not None:
        nuclear_area_floor = float(numpy.percentile(measurements["nucleus_polygon_area"], NUCLEAR_AREA_FLOOR_PERCENTILE))
        measurements = measurements[measurements["nucleus_polygon_area"] >= nuclear_area_floor]
        record_step(f"cells with nuclear area >= {nuclear_area_floor:.2f} µm² (p{NUCLEAR_AREA_FLOOR_PERCENTILE:g})", len(measurements))

    # ---- 6. Minimum counts per cell ----------------------------------------------------------
    row_of_cell = pandas.Series(numpy.arange(len(table_cell_ids)), index=table_cell_ids)
    total_gene_counts = numpy.asarray(counts.sum(axis=1)).ravel()
    measurements["total_gene_counts"] = total_gene_counts[row_of_cell.loc[measurements.index].to_numpy()]
    measurements = measurements[measurements["total_gene_counts"] >= MINIMUM_COUNTS_PER_CELL]
    record_step(f"cells with >= {MINIMUM_COUNTS_PER_CELL} gene counts", len(measurements))

    # ---- 7. Build the per-cell table that R reads --------------------------------------------
    kept_cell_ids = measurements.index.to_numpy()
    cell_table = pandas.DataFrame({
        "cell_id": kept_cell_ids,
        "centroid_x": cell_information.loc[kept_cell_ids, "x_centroid"].to_numpy(dtype=float),
        "centroid_y": cell_information.loc[kept_cell_ids, "y_centroid"].to_numpy(dtype=float),
        "cell_area": cell_information.loc[kept_cell_ids, "cell_area"].to_numpy(dtype=float),
        "xenium_nucleus_area": (cell_information.loc[kept_cell_ids, "nucleus_area"].to_numpy(dtype=float)
                                if "nucleus_area" in cell_information.columns else numpy.nan),
        "nucleus_polygon_area": measurements["nucleus_polygon_area"].to_numpy(),
        "number_of_nucleus_outlines": measurements["number_of_nucleus_outlines"].to_numpy(),
        "total_gene_counts": measurements["total_gene_counts"].to_numpy(),
    })
    for metric_name in metrics.SHAPE_METRIC_NAMES:
        cell_table[metric_name] = measurements[metric_name].to_numpy()

    # Optional cell-type labels (pancreas). MechanoMap cell names carry a tile prefix that is removed.
    if settings["cell_type_file"]:
        annotated = anndata.read_h5ad(INPUT_DATA_FOLDER / settings["cell_type_file"], backed="r")
        annotation = annotated.obs[[settings["cell_type_column"]]].copy()
        annotation.index = annotated.obs_names.str.replace(TILE_PREFIX_PATTERN, "", regex=True)
        annotation = annotation[~annotation.index.duplicated()]
        cell_table["cell_type"] = annotation[settings["cell_type_column"]].astype(str).reindex(kept_cell_ids).to_numpy()
        number_labelled = cell_table["cell_type"].notna().sum()
        print(f"  cell-type labels found for {number_labelled:,} of {len(cell_table):,} cells "
              f"(column '{settings['cell_type_column']}')")
        annotated.file.close()

    # ---- 8. Write everything -----------------------------------------------------------------
    kept_rows = row_of_cell.loc[kept_cell_ids].to_numpy()
    kept_counts = counts[kept_rows, :].tocoo()   # cells x genes, same row order as cell_table

    scipy.io.mmwrite(str(prep_folder / "counts.mtx"), kept_counts, field="integer",
                     comment=f" {PROPRIETARY_NOTICE}\n rows = cells (counts_cells.txt), columns = genes (counts_genes.txt)")
    write_list_with_notice(kept_cell_ids, prep_folder / "counts_cells.txt")
    write_list_with_notice(gene_names, prep_folder / "counts_genes.txt")
    write_csv_with_notice(cell_table, prep_folder / "cells.csv", index=False)
    write_csv_with_notice(pandas.DataFrame(quality_control_steps), prep_folder / "prep_qc_summary.csv", index=False)

    prep_metadata = {
        "proprietary_notice": PROPRIETARY_NOTICE,
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "notebook": "notebooks/nuclear_shape_prep.ipynb",
        "dataset": dataset_name,
        "zarr_store": str(zarr_path),
        "coordinate_units": "microns",
        "nucleus_outlines": "zarr shapes/nucleus_boundaries (Xenium)",
        "expression": "zarr tables/table (cell_feature_matrix.h5), Gene Expression features only",
        "shape_metric_definitions": {
            "solidity": "area / convex hull area",
            "extent": "area / axis-aligned bounding box area",
            "shape_index": "perimeter / sqrt(area)",
            "circularity": "4 * pi * area / perimeter^2",
            "perimeter": "outline length (microns)",
            "axis_ratio": "minor / major axis of the equivalent ellipse (second moments of area)",
        },
        "measurement_method": "exact polygon geometry (shapely); largest piece kept, holes filled",
        "multiple_nuclei_rule": MULTIPLE_NUCLEI_RULE,
        "nuclear_area_floor_percentile": NUCLEAR_AREA_FLOOR_PERCENTILE,
        "nuclear_area_floor_square_microns": nuclear_area_floor,
        "minimum_counts_per_cell": MINIMUM_COUNTS_PER_CELL,
        "cell_type_file": settings["cell_type_file"],
        "cell_type_column": settings["cell_type_column"],
        "number_of_cells_exported": int(len(cell_table)),
        "number_of_genes_exported": int(len(gene_names)),
        "quality_control_steps": quality_control_steps,
        "python_version": platform.python_version(),
        "package_versions": {"spatialdata": spatialdata.__version__, "anndata": anndata.__version__,
                             "numpy": numpy.__version__, "pandas": pandas.__version__},
    }
    with open(prep_folder / "prep_metadata.json", "w", encoding="utf-8") as metadata_file:
        json.dump(prep_metadata, metadata_file, indent=2, ensure_ascii=False)

    print(f"  wrote {len(cell_table):,} cells x {len(gene_names)} genes to {prep_folder}")
    display(cell_table[metrics.SHAPE_METRIC_NAMES].describe().T)


========== pancreas: Xenium_V1_human_Pancreas_FFPE_outs.zarr ==========


c:\Users\domin\miniforge3\envs\segbench\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  count table: 140,702 cells x 377 genes (control features removed)
  cells in the zarr count table                             140,702 cells
  cells with at least one nucleus outline                   136,531 cells
  invalid / zero-area nucleus outlines dropped: 5,444
  cells with at least one valid nucleus outline             131,087 cells
  cells with more than one valid nucleus outline: 0
  cells after the 'largest' multiple-nuclei rule            131,087 cells
  cells with all six metrics measurable                     131,087 cells
  cells with nuclear area >= 4.36 µm² (p1)                  129,776 cells
  cells with >= 10 gene counts                              112,909 cells
  cell-type labels found for 112,909 of 112,909 cells (column 'cell_type')
  wrote 112,909 cells x 377 genes to c:\MechanomeBioRepos\nuclear_shape\outputs\pancreas\prep


C:\Users\domin\AppData\Local\Temp\ipykernel_17164\2322105650.py:142: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  "package_versions": {"spatialdata": spatialdata.__version__, "anndata": anndata.__version__,


,count,mean,std,min,25%,50%,75%,max
solidity,112909.0,0.946647,0.053407,0.394383,0.934362,0.965272,0.980107,1.000000
extent,112909.0,0.681183,0.115229,0.087919,0.627693,0.711590,0.765641,0.914533
shape_index,112909.0,4.108771,0.535541,3.581314,3.768308,3.930621,4.245686,12.425084
circularity,112909.0,0.772963,0.147764,0.081398,0.697131,0.813369,0.884946,0.979772
perimeter,112909.0,19.789887,6.320685,7.731891,15.678284,19.146618,22.347521,88.449913
axis_ratio,112909.0,0.634584,0.202863,0.067940,0.489830,0.653451,0.800621,0.999160



========== lung: Xenium_V1_humanLung_Cancer_FFPE_outs.zarr ==========
  count table: 162,254 cells x 377 genes (control features removed)
  cells in the zarr count table                             162,254 cells
  cells with at least one nucleus outline                   156,628 cells
  invalid / zero-area nucleus outlines dropped: 8,146
  cells with at least one valid nucleus outline             148,482 cells
  cells with more than one valid nucleus outline: 0
  cells after the 'largest' multiple-nuclei rule            148,482 cells
  cells with all six metrics measurable                     148,482 cells
  cells with nuclear area >= 4.79 µm² (p1)                  146,997 cells
  cells with >= 10 gene counts                              140,259 cells
  wrote 140,259 cells x 377 genes to c:\MechanomeBioRepos\nuclear_shape\outputs\lung\prep


C:\Users\domin\AppData\Local\Temp\ipykernel_17164\2322105650.py:142: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  "package_versions": {"spatialdata": spatialdata.__version__, "anndata": anndata.__version__,


,count,mean,std,min,25%,50%,75%,max
solidity,140259.0,0.946996,0.052509,0.419777,0.934345,0.965432,0.979715,1.000000
extent,140259.0,0.680123,0.101740,0.078075,0.628004,0.702505,0.754401,0.937492
shape_index,140259.0,4.076648,0.431394,3.568232,3.797922,3.947292,4.205981,10.535112
circularity,140259.0,0.776063,0.126636,0.113222,0.710355,0.806513,0.871200,0.986969
perimeter,140259.0,22.031233,7.459796,8.156625,16.434034,20.790427,26.859211,89.905444
axis_ratio,140259.0,0.624616,0.172678,0.060319,0.503052,0.634167,0.755705,0.998780


### Next step

Open `nuclear_shape_gsem.ipynb` **in WSL** (see the instructions at the top of that notebook). It reads, for each tissue, from `outputs/<tissue>/prep/`:

- `cells.csv` — one row per cell: position, cell area, the six shape metrics (and `cell_type` for pancreas)
- `counts.mtx`, `counts_cells.txt`, `counts_genes.txt` — the raw cells × genes count matrix
- `prep_qc_summary.csv`, `prep_metadata.json` — how many cells each filter removed, and every setting used

© 2026 Mechanome Bio. Proprietary and confidential.